# Prévisions de référence : des règles simples

## Pourquoi commencer par des baselines ?

Un modèle complexe n’est utile que s’il fait mieux que des règles faciles à comprendre. Ce notebook mesure deux points de comparaison sur les mêmes séries :

- **Naïve** : le prix prévu reprend le dernier prix connu.
- **Saisonnière** : le prix prévu reprend celui observé à la même période l’année précédente.

On évalue les horizons de 1, 2 et 3 mois. Les erreurs sont calculées sur les prix observés, avec puis sans les périodes où beaucoup de prix restent identiques.

## 1. Charger le panel et repérer les mois à surveiller

Le panel brut conserve les prix source. La fonction `bad_months` repère les mois où plus de la moitié des séries n’a pas changé, ainsi que le mois suivant, potentiellement touché par un rattrapage des relevés.

In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.evaluation.baselines import load_panel, bad_months, evaluate, summarize

panel = load_panel(brut=True)
print(sorted(d.strftime("%Y-%m") for d in bad_months(panel)))   # mois jugés non fiables

['2023-11', '2023-12', '2024-01', '2024-05', '2024-06', '2024-09', '2024-10', '2024-11', '2024-12']


## 2. Comparer les méthodes et les horizons

Pour chaque horizon, on calcule l’erreur absolue moyenne (MAE) et l’erreur absolue moyenne en pourcentage (MAPE), avec puis sans les mois jugés peu fiables. Une MAPE plus faible indique une meilleure précision moyenne en pourcentage.

In [2]:
rows = []
for h in [1, 2, 3]:
    for method in ["naive", "seasonal"]:
        for excl in [False, True]:
            s = summarize(evaluate(panel, h, method, exclude_bad=excl)).iloc[0]
            rows.append({"h": h, "méthode": method, "sans_mois_figés": excl, **s.round(1).to_dict()})
print(pd.DataFrame(rows).to_string(index=False))

 h  méthode  sans_mois_figés  MAE  MAPE_%       n
 1    naive            False 39.8    10.0 16291.0
 1    naive             True 41.6    10.7 12789.0
 1 seasonal            False 99.9    23.8 16300.0
 1 seasonal             True 95.5    24.4 11612.0
 2    naive            False 57.6    14.5 16258.0
 2    naive             True 60.8    15.8 11903.0
 2 seasonal            False 99.9    23.8 16300.0
 2 seasonal             True 95.5    24.4 11612.0
 3    naive            False 70.8    17.8 16251.0
 3    naive             True 74.9    19.6 11611.0
 3 seasonal            False 99.9    23.8 16300.0
 3 seasonal             True 95.5    24.4 11612.0


In [3]:
# Détail par année pour la naïve à 1 mois, avec et sans mois figés
print(summarize(evaluate(panel, 1, "naive"), by_year=True).round(1))
print(summarize(evaluate(panel, 1, "naive", exclude_bad=True), by_year=True).round(1))

       MAE  MAPE_%     n
date                    
2022  38.5    10.8  3582
2023  31.8     7.4  3501
2024  32.0     7.1  3496
2025  56.1    12.7  3495
2026  41.3    12.8  2217
       MAE  MAPE_%     n
date                    
2022  38.5    10.8  3582
2023  34.1     7.8  2908
2024  32.6     7.8   868
2025  54.4    12.5  3214
2026  41.3    12.8  2217


## 3. Examiner les résultats par année

La dernière comparaison détaille la prévision naïve à un mois année par année. Elle permet de voir si l’erreur est stable ou si certaines périodes, notamment celles avec des prix figés, pèsent davantage dans le résultat.